# Infrastructures

We can simply adopt some of them from [the main top-level
page](https://amane-fujimiya.github.io/Research/tutorial.html), however,
clarify about current CI/CD template is needed, especially for contents
hosted on such format (which is preferable since it is more organized).
So is for some or even more of our templates and projects on GitLab.

### The GitLab CI/CD

For now, we have completed the configuration as followed (not every
preprocessors are there, but the one needed of).

``` yml
stages:
  - deploy

pages:
  stage: deploy
  image: rust:latest
  variables:
    MDBOOK_VERSION: 0.4.52
    MDBOOK_BIB_VERSION: 0.0.7
    CARGO_HOME: $CI_PROJECT_DIR/cargo
  before_script:
    - mkdir -p "$CARGO_HOME/bin"
    - export PATH="$PATH:$CARGO_HOME/bin"
    # Download prebuilt mdBook binary, but do not overwrite existing files
    - curl -LsSf "https://github.com/rust-lang/mdBook/releases/download/v${MDBOOK_VERSION}/mdbook-v${MDBOOK_VERSION}-x86_64-unknown-linux-gnu.tar.gz" | tar -xzf - --skip-old-files
    # Optionally download prebuilt mdbook-bib (commented out)
    #- curl -LsSf "https://github.com/francisco-perez-sorrosal/mdbook-bib/releases/download/v${MDBOOK_BIB_VERSION}/mdbook-bib-v${MDBOOK_BIB_VERSION}-linux-amd64.tar.gz" | tar -xzf - --skip-old-files

    # Helper to install a cargo binary only if it's missing
    - |
      install_if_missing() {
        crate="$1"
        bin="$2" # optional override for the binary name
        if [ -z "$bin" ]; then
          bin="$crate"
        fi
        if command -v "$bin" >/dev/null 2>&1; then
          echo "Skipping $crate: $bin already in PATH"
        else
          echo "Installing $crate ..."
          cargo install "$crate"
        fi
      }

    # Install plugins only if missing
    - install_if_missing mdbook-katex
    - install_if_missing mdbook-toc
    - install_if_missing mdbook-footnote
    - install_if_missing mdbook-numeq
    - install_if_missing mdbook-numthm
    - install_if_missing mdbook-image-size
    - install_if_missing mdbook-bib

  script:
    - RUST_LOG=mdbook_bib=trace ./mdbook build docs -d public
    - mv docs/public public
  rules:
    - if: $CI_COMMIT_REF_NAME == $CI_DEFAULT_BRANCH
  artifacts:
    paths:
      - public
  cache:
    key: "cargo-cache-v1"
    paths:
      - $CARGO_HOME/registry
      - $CARGO_HOME/git
      - $CARGO_HOME/bin
```

It should be good for now, except issues with compatibility for `cargo`
and Rust versions with `mdbook`. Since there exists stricter existence
in memory window for later version, `mdbook` can fail compiling, thus is
why I use pre-compiled binary for certain preprocessors that require
version compatibility.

### Troublesome issues

There exists an issue for now, that it feels like it is hating me for no
reason. So, `mdbook-katex` is what to use if you want compatibility with
older syntax in `mdbook`, i.e. `$$` maths. In one of the repository, I
have:

``` markdown
\\[
\vec{L} = 
\det 
\begin{bmatrix}
    \vec{i}  & \vec{j} & \vec{k} \\\\
    \hat{x} & \hat{y} & \hat{z} \\\\
    \hat{p_{x}} & \hat{p_{y}} & \hat{p_{z}}
\end{bmatrix}
\\]
```

in which, if you go for

    \\[
    \vec{L} = 
    \det 
    \begin{bmatrix}
        \vec{i}  & \vec{j} & \vec{k} \\\\
        \hat{x} & \hat{y} & \hat{z} \\\\
        \hat{p}_{x} & \hat{p}_{y} & \hat{p}_{z}
    \end{bmatrix}
    \\]

The parser will fail. I don’t really know why, but the `mdbook`
processor is really brittle. Also, should it be recommended, please
change from the old `$$` or `$$[...]$$` to the `mdbook` syntax if such
issue arise.